# Atomic Orbitals

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LightSource
import ipywidgets as widgets
from IPython.display import display

## Special Functions

In [ ]:
def assoc_laguerre(q, p, x):
# Associated Laguerre polynomial L_q^p(x).
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    for i in range(q + 1):
        coef = ((-1) ** i) * math.comb(q + p, q - i) / math.factorial(i)
        result = result + coef * x ** i
    return result

def assoc_legendre(l, m, x):
# Associated Legendre P_l^m(x), m >= 0, via stable upward recurrence.
    x = np.asarray(x, dtype=float)
    m = abs(m)
    pmm = np.ones_like(x)
    if m > 0:
        somx2 = np.sqrt((1 - x) * (1 + x))
        fact = 1.0
        for _ in range(1, m + 1):
            pmm = pmm * (-fact) * somx2
            fact += 2.0
    if l == m:
        return pmm
    pmmp1 = x * (2 * m + 1) * pmm
    if l == m + 1:
        return pmmp1
    pll = np.zeros_like(x)
    for ll in range(m + 2, l + 1):
        pll = (x * (2 * ll - 1) * pmmp1 - (ll + m - 1) * pmm) / (ll - m)
        pmm, pmmp1 = pmmp1, pll
    return pll


In [ ]:
a = 1.0  # Bohr radius, atomic units

## Hydrogen Wavefunction (Pieces/ Parts)

In [ ]:
def radial_wavefunction(n, l, r):
    rho = 2 * r / (n * a)
    norm = math.sqrt(
        (2.0 / (n * a)) ** 3 * math.factorial(n - l - 1) / (2.0 * n * math.factorial(n + l))
    )
    return norm * np.exp(-rho / 2) * rho ** l * assoc_laguerre(n - l - 1, 2 * l + 1, rho)


def angular_real(l, m, theta, phi):
    """
    REAL angular function (the familiar textbook p_x, p_y, d_xy ... shapes).
    m = 0 -> Y_l0 ;  m > 0 -> cos(m phi) type ;  m < 0 -> sin(|m| phi) type.
    """
    mm = abs(m)
    norm = math.sqrt((2 * l + 1) / (4 * np.pi) * math.factorial(l - mm) / math.factorial(l + mm))
    P = assoc_legendre(l, mm, np.cos(theta))
    if m == 0:
        return norm * P
    trig = np.cos(mm * phi) if m > 0 else np.sin(mm * phi)
    return math.sqrt(2.0) * norm * P * trig

In [ ]:
SUBSHELL = ['s', 'p', 'd', 'f', 'g', 'h', 'i']

In [ ]:
def orbital_name(n, l, m):
    letter = SUBSHELL[l] if l < len(SUBSHELL) else f"l{l}"
    names = {
        (1, 0): "z", (1, 1): "x", (1, -1): "y",
        (2, 0): "z²", (2, 1): "xz", (2, -1): "yz", (2, 2): "x²-y²", (2, -2): "xy",
    }
    tag = names.get((l, m), "")
    return f"{n}{letter}" + (f"$_{{{tag}}}$" if tag else f" (m={m:+d})")

## Color Maps and Themes

In [ ]:
BG = "#070b17"
POS_CMAP = LinearSegmentedColormap.from_list(
    "pos", ["#5b0a91", "#d11f8a", "#ff5e3a", "#ffb000", "#fff2a8"])
NEG_CMAP = LinearSegmentedColormap.from_list(
    "neg", ["#0a1a6e", "#1457d6", "#00b4ff", "#00f0d0", "#d9fff7"])

In [ ]:
def _style_axes(ax, lim, elev, azim):
    ax.set_facecolor(BG)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()
    # glowing coordinate axes through the nucleus
    for (dx, dy, dz), lab in (((1, 0, 0), "x"), ((0, 1, 0), "y"), ((0, 0, 1), "z")):
        ax.plot([-lim * dx, lim * dx], [-lim * dy, lim * dy], [-lim * dz, lim * dz],
                color="#7f8aa8", lw=0.8, alpha=0.6)
        ax.text(1.08 * lim * dx, 1.08 * lim * dy, 1.08 * lim * dz, lab,
                color="#cfd8ff", fontsize=11, ha="center", va="center")
    ax.scatter([0], [0], [0], s=25, color="white", alpha=0.9)  # nucleus

## Boundary Surface

In [ ]:
def boundary_surface(n, l, m, thr=0.02, n_theta=90, n_phi=180, n_r=320):
    rmax = 2.5 * n ** 2 + 8
    r = np.linspace(1e-3, rmax, n_r)
    theta = np.linspace(0, np.pi, n_theta)
    phi = np.linspace(0, 2 * np.pi, n_phi)
    TH, PH = np.meshgrid(theta, phi, indexing="ij")

    R = radial_wavefunction(n, l, r)           # (n_r,)
    A = angular_real(l, m, TH, PH)             # (n_theta, n_phi)

    dens = (A ** 2)[:, :, None] * (R ** 2)[None, None, :]
    mask = dens >= thr * dens.max()
    has = mask.any(axis=2)
    idx = n_r - 1 - np.argmax(mask[:, :, ::-1], axis=2)   # outermost True
    r_edge = np.where(has, r[idx], 0.0)

    sign = np.sign(R[idx]) * np.sign(A)
    sign[sign == 0] = 1
    X = r_edge * np.sin(TH) * np.cos(PH)
    Y = r_edge * np.sin(TH) * np.sin(PH)
    Z = r_edge * np.cos(TH)

    # colour: hue family from sign, brightness from angular strength
    t = 0.15 + 0.85 * np.abs(A) / (np.abs(A).max() + 1e-12)
    colors = np.where((sign > 0)[..., None], POS_CMAP(t), NEG_CMAP(t))
    return X, Y, Z, colors, r_edge.max()

In [ ]:
def draw_surface(ax, n, l, m, elev, azim):
    X, Y, Z, colors, rm = boundary_surface(n, l, m)
    ls = LightSource(azdeg=220, altdeg=50)
    ax.plot_surface(X, Y, Z, facecolors=colors, rstride=1, cstride=1,
                    linewidth=0, antialiased=True, shade=True, lightsource=ls)
    lim = rm * 1.15
    _style_axes(ax, lim, elev, azim)
    ax.set_title("Boundary surface", color="#cfd8ff", fontsize=11, pad=-6)
    return lim

## Probability Cloud

In [ ]:
def cloud_points(n, l, m, n_points=14000, seed=None):
    rng = np.random.default_rng(seed)
    rmax = 2.5 * n ** 2 + 8
    rg = np.linspace(1e-4, rmax, 4000)
    Rg = radial_wavefunction(n, l, rg)
    pdf = rg ** 2 * Rg ** 2
    cdf = np.cumsum(pdf); cdf /= cdf[-1]

    # angular max for rejection
    tt, pp = np.meshgrid(np.linspace(0, np.pi, 120), np.linspace(0, 2 * np.pi, 240), indexing="ij")
    amax = (angular_real(l, m, tt, pp) ** 2).max()

    pts = []
    got = 0
    while got < n_points:
        N = 40000
        cth = rng.uniform(-1, 1, N)
        ph = rng.uniform(0, 2 * np.pi, N)
        th = np.arccos(cth)
        A = angular_real(l, m, th, ph)
        keep = rng.uniform(0, 1, N) < A ** 2 / amax
        th, ph, A = th[keep], ph[keep], A[keep]
        rr = np.interp(rng.uniform(0, 1, th.size), cdf, rg)
        s = np.sign(radial_wavefunction(n, l, rr)) * np.sign(A)
        pts.append(np.column_stack([rr * np.sin(th) * np.cos(ph),
                                    rr * np.sin(th) * np.sin(ph),
                                    rr * np.cos(th), rr, s]))
        got += th.size
    pts = np.vstack(pts)[:n_points]
    return pts

In [ ]:
def draw_cloud(ax, n, l, m, elev, azim):
    P = cloud_points(n, l, m, seed=1)
    x, y, z, r, s = P.T
    # soft glow: brighter near the nucleus, fading outwards
    t = np.clip(1.0 - r / (np.percentile(r, 99) + 1e-9), 0, 1) ** 0.6 * 0.75 + 0.25
    cols = np.where((s > 0)[:, None], POS_CMAP(t), NEG_CMAP(t))
    ax.scatter(x, y, z, c=cols, s=3.5, alpha=0.55, linewidths=0, depthshade=True)
    lim = np.percentile(r, 99.5) * 1.1
    lim = max(lim, 2.0)
    _style_axes(ax, lim, elev, azim)
    ax.set_title("Probability cloud", color="#cfd8ff", fontsize=11, pad=-6)
    return lim

## Main Plot Function

In [ ]:
def plot_orbital_3d(n, l, m, view="Surface", elev=22, azim=-55):
    two = view == "Both"
    fig = plt.figure(figsize=(11.5 if two else 6.2, 6.0), facecolor=BG)

    if view in ("Surface", "Both"):
        ax1 = fig.add_subplot(1, 2 if two else 1, 1, projection="3d")
        draw_surface(ax1, n, l, m, elev, azim)
    if view in ("Cloud", "Both"):
        ax2 = fig.add_subplot(1, 2 if two else 1, 2 if two else 1, projection="3d")
        draw_cloud(ax2, n, l, m, elev, azim)

    fig.suptitle(f"{orbital_name(n, l, m)} orbital   (n={n}, l={l}, m={m:+d})\n"
                 f"radial nodes = {n - l - 1},  angular nodes = {l}",
                 color="white", fontsize=13, y=0.97)
    fig.text(0.5, 0.03, "gold/magenta:  ψ > 0        cyan/blue:  ψ < 0",
             color="#cfd8ff", ha="center", fontsize=10)
    plt.subplots_adjust(left=0, right=1, top=0.9, bottom=0.06, wspace=0)
    plt.show()


In [ ]:
n_slider = widgets.IntSlider(value=3, min=1, max=6, step=1, description="n:")
l_slider = widgets.IntSlider(value=2, min=0, max=2, step=1, description="l:")
m_slider = widgets.IntSlider(value=0, min=-2, max=2, step=1, description="m:")
view_dd = widgets.Dropdown(options=["Surface", "Cloud", "Both"], value="Surface",
                           description="View:")
elev_slider = widgets.IntSlider(value=22, min=-90, max=90, step=2, description="Elev:")
azim_slider = widgets.IntSlider(value=-55, min=-180, max=180, step=5, description="Azim:")


def _update_l_range(*args):
    l_slider.max = n_slider.value - 1
    if l_slider.value > l_slider.max:
        l_slider.value = l_slider.max


def _update_m_range(*args):
    m_slider.min = -l_slider.value
    m_slider.max = l_slider.value
    m_slider.value = max(m_slider.min, min(m_slider.max, m_slider.value))


n_slider.observe(_update_l_range, names="value")
l_slider.observe(_update_m_range, names="value")

ui = widgets.VBox([widgets.HBox([n_slider, l_slider, m_slider]),
                   widgets.HBox([view_dd, elev_slider, azim_slider])])
out = widgets.interactive_output(
    plot_orbital_3d,
    {"n": n_slider, "l": l_slider, "m": m_slider, "view": view_dd,
     "elev": elev_slider, "azim": azim_slider})


In [ ]:
display(ui, out)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 620x600 with 1 Axes>', 'i…